In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt

from content_based import prepare_metadata_soup, calculate_weighted_rating

In [ ]:
movies = pd.read_csv("../data/raw/movies.csv")
ratings = pd.read_csv("../data/raw/ratings.csv")
tags = pd.read_csv("../data/raw/tags.csv")

In [ ]:
movies_soup = prepare_metadata_soup(movies=movies, tags=tags)

movies_soup.head()

In [ ]:
movies_soup["decade"].value_counts().sort_index()

In [ ]:
movies_soup["decade"].value_counts().sort_index().plot(kind="bar")

plt.title("Movies by Decade")
plt.xlabel("Decade")
plt.ylabel("Number of Movies")
plt.show()

In [ ]:
movies_soup[["title", "genres", "soup_genres"]].head(10)

In [ ]:
movies_soup[movies_soup["genres"] == "(no genres listed)"][
    ["title", "genres", "soup_genres"]
].head()

In [ ]:
movies_soup[["title", "soup_tags"]].head(10)

In [ ]:
movies_with_tags = tags["movieId"].nunique()

total_movies = movies["movieId"].nunique()

movies_without_tags = total_movies - movies_with_tags

movies_without_tags

In [ ]:
movies_soup[movies_soup["tags_joined"].isnull()][
    ["title", "soup_genres", "soup_tags"]
].head(10)

In [ ]:
movies_model = calculate_weighted_rating(
    movies=movies_soup,
    ratings=ratings,
    popularity_percentile=0.70
)

movies_model.head()

In [ ]:
movies_model[
    [
        "movieId",
        "title",
        "v",
        "R",
        "weighted_rating",
        "quality_score"
    ]
].head(10)

In [ ]:
movies_model["quality_score"].describe()

In [ ]:
movies_model["quality_score"].min(), movies_model["quality_score"].max()

In [ ]:
movies_model["quality_score"].hist(bins=30)

plt.title("Quality Score Distribution")
plt.xlabel("Quality Score")
plt.ylabel("Number of Movies")
plt.show()

In [ ]:
movies_model.sort_values(
    by="quality_score",
    ascending=False
)[
    ["title", "v", "R", "weighted_rating", "quality_score"]
].head(10)

In [ ]:
import os

os.makedirs("../data/features", exist_ok=True)

In [ ]:
movies_model.to_csv("../data/features/movies_content_features.csv", index=False)

## Feature Engineering Observations

- The release year was extracted from the movie title and transformed into a decade label.
- Genres were cleaned and combined with the decade to create `soup_genres`.
- User-generated tags were grouped by movieId to create `soup_tags`.
- Movies without tags were assigned their genre-based metadata soup as fallback text.
- A weighted rating score was calculated using rating count, average rating, global mean rating, and a popularity threshold.
- The weighted rating was normalized into `quality_score`, ranging from 0 to 1.
- The resulting feature dataset was saved as `data/features/movies_content_features.csv`.